Download data from [UK-based online retail](https://archive.ics.uci.edu/dataset/502/online+retail+ii).
future is 369 days

# Cleaning

## Importing Libraries and data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#from tqdm import tqdm
#import re
import os
#from tqdm.notebook import tqdm
from tqdm.auto import tqdm
from collections import defaultdict

#from sentence_transformers import SentenceTransformer
#from sklearn.cluster import KMeans,MiniBatchKMeans, AgglomerativeClustering
#from sklearn.metrics import silhouette_score

In [2]:
print(os.getcwd())

# reading and combining data from excel sheets
#df_2010 = pd.read_excel("data/online_retail_II.xlsx", sheet_name="Year 2009-2010")
#df_2011 = pd.read_excel("data/online_retail_II.xlsx", sheet_name="Year 2010-2011")
#df = pd.concat([df_2010, df_2011], ignore_index=True)

#df.to_csv("data/online_retail.csv", index=False)

# reading previously saved csv, which contains combined data of two excel sheets
df_full = pd.read_csv("data/online_retail.csv")

/Users/taniazamansarna/projects/customer-clv


In [3]:
df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [4]:
df_full.head(4)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom


In [5]:
df_full['Invoice'] = df_full['Invoice'].astype(str).str.upper().str.strip()
df_full['InvoiceDate'] = pd.to_datetime(df_full['InvoiceDate'])
df_full = df_full.rename(columns={"Customer ID": "Customer_ID"})
df_full["Customer_ID"] = df_full["Customer_ID"].astype("Int64")
df_full['StockCode'] = df_full['StockCode'].astype(str).str.upper().str.strip()
df_full['Description'] = df_full['Description'].astype(str).str.upper().str.strip()
df_full['Country'] = df_full['Country'].astype(str).str.upper().str.strip()

df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer_ID  824364 non-null   Int64         
 7   Country      1067371 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 66.2 MB


In [6]:
# Deleting lines that put as a test, or as fee	
df_full = df_full[~(df_full["Description"]=="CHECKED")]
df_full = df_full[~(df_full["Description"]=="CHECK")]
df_full = df_full[~(df_full["Description"]=="CHECK?")]
df_full = df_full[~(df_full["Description"]=="STOCK CHECK")]
df_full = df_full[~(df_full["Description"]=="CARRIAGE")]

df_full = df_full[~df_full["Description"].str.contains("damaged", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("BANK CHARGES", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("DISCOUNT", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("POSTAGE", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("MANUAL", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("ADJUST", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("test", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("lost", case=False, na=False)]
df_full = df_full[~df_full["Description"].str.contains("samples", case=False, na=False)]

# 5. Remove administrative/service stock codes
non_product_codes = ['DOT', 'POST', 'C2', 'M', 'BANK CHARGES', 'PADS', 'CRUK']
df_full = df_full[~df_full['StockCode'].isin(non_product_codes)]

df_full = df_full[~df_full["StockCode"].str.contains("test", case=False, na=False)]
df_full = df_full[~df_full["StockCode"].str.contains("fee", case=False, na=False)]

In [7]:
df_full.info()

<class 'pandas.DataFrame'>
Index: 1061204 entries, 0 to 1067369
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1061204 non-null  str           
 1   StockCode    1061204 non-null  str           
 2   Description  1056835 non-null  str           
 3   Quantity     1061204 non-null  int64         
 4   InvoiceDate  1061204 non-null  datetime64[us]
 5   Price        1061204 non-null  float64       
 6   Customer_ID  820628 non-null   Int64         
 7   Country      1061204 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 73.9 MB


## Splitting the data into calibration and holdout periods

Customer Lifetime Value (CLV) models should always be evaluated on future customer behavior rather than the same data used for model development. To accomplish this, the transaction history is divided into two non-overlapping time periods:

- **Calibration period:** Used to estimate customer behavior and fit the predictive models.
- **Holdout period:** Used only for evaluation. Transactions in this period are treated as unseen future data and are used to assess how well the models generalize.

In this project, **5 December 2010** is selected as the split date.

- Transactions **before 5 December 2010** belong to the **calibration period**.
- Transactions **on or after 5 December 2010** belong to the **holdout period**.

Performing this split immediately after loading the dataset ensures that all subsequent data cleaning, feature engineering, and model evaluation are conducted independently within each period. This avoids any information leakage from the future (holdout period) into the past (calibration period), resulting in a more realistic evaluation of customer lifetime value predictions.

In [8]:
# Split date
split_date = pd.Timestamp("2010-12-05")

# Calibration datasets
df = df_full[df_full["InvoiceDate"] <= split_date].copy()

print(f"Calibration transactions: {len(df):,}")
print(f"Holdout transactions   :  {len(df_full[df_full["InvoiceDate"] > split_date]):,}")

Calibration transactions: 514,851
Holdout transactions   :  546,353


# Identifying and handling cancelled/ returned order

This step assigns a clear transaction status to each line item by matching negative-quantity rows against earlier positive-quantity rows from the same customer, product, quantity, and price.

The idea is simple:

- A **positive** quantity usually means a normal purchase.
- A **negative** quantity usually means a return, cancellation, or reversal.
- If a negative row can be matched to a previous positive row with the same customer, stock code, absolute quantity, and price, then the two rows belong together.
- The positive row is marked as **`cancelled`**.
- The negative row is marked as **`return: purchase found`**.
- If no matching purchase exists, the negative row is marked as **`return: purchase not found`**.
- Any remaining positive rows that were never matched to a return are marked as **`completed`**.

This approach is much faster than filtering the whole dataframe for every negative row. Instead, the data is processed once in chronological order, and a hash-based lookup structure stores unmatched positive rows. That makes matching efficient even for large datasets.

The `status` column is a new descriptive field that summarizes the outcome of the matching logic for each row. It is useful later for cleaning, analysis, CLV modeling, and churn/retention work because it separates normal purchases from returns and cancellations.

In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 514851 entries, 0 to 532879
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      514851 non-null  str           
 1   StockCode    514851 non-null  str           
 2   Description  511953 non-null  str           
 3   Quantity     514851 non-null  int64         
 4   InvoiceDate  514851 non-null  datetime64[us]
 5   Price        514851 non-null  float64       
 6   Customer_ID  411201 non-null  Int64         
 7   Country      514851 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 35.8 MB


In [10]:
# negative price also mean a cancellation of order; 
# Since in this note, we standardize cancellation of ordre by -ve Quantity, we follow that convention
df['Quantity'] = df['Quantity'] * np.sign(df['Price'])

In [11]:
# Sort by time
df = df.sort_values("InvoiceDate").reset_index(drop=True)

# Status column
df["status"] = pd.NA

# ---------------------------------------------------
# Matching structure:
# key = (Customer_ID, StockCode, abs(Quantity), Price)
# value = list of unmatched positive row positions
# ---------------------------------------------------
open_sales = defaultdict(list)

# Use tqdm for progress tracking
for i, row in tqdm(
    enumerate(df.itertuples(index=False)),
    total=len(df),
    desc="Matching returns"
):
    key = (
        row.Customer_ID,
        row.StockCode,
        abs(row.Quantity),
        row.Price
    )
    if row.Quantity > 0:
        open_sales[key].append(i)

    elif row.Quantity < 0:
        if open_sales[key]:
            match_i = open_sales[key].pop()
            df.at[match_i, "status"] = "cancelled"
            df.at[i, "status"] = "return: purchase found"
        else:
            df.at[i, "status"] = "return: purchase not found"

# Any remaining positive rows are completed purchases
df.loc[(df["Quantity"] > 0) & (df["status"].isna()), "status"] = "completed"

Matching returns:   0%|          | 0/514851 [00:00<?, ?it/s]

In [12]:
df['status'] = df['status'].astype(str)
df.info()

## Keeping only completed orders
df = df[df['status']=='completed']
#df_return = df[df['status'] == 'return: purchase found']

print(f'Size of the calibration data after adjusting returned order is: {len(df)}')

<class 'pandas.DataFrame'>
RangeIndex: 514851 entries, 0 to 514850
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      514851 non-null  str           
 1   StockCode    514851 non-null  str           
 2   Description  511953 non-null  str           
 3   Quantity     514851 non-null  float64       
 4   InvoiceDate  514851 non-null  datetime64[us]
 5   Price        514851 non-null  float64       
 6   Customer_ID  411201 non-null  Int64         
 7   Country      514851 non-null  str           
 8   status       511325 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(2), str(5)
memory usage: 35.8 MB
Size of the calibration data after adjusting returned order is: 498790


In [13]:
# droping status column before saving to csv
df.drop(columns = 'status', inplace=True)
#df_return.drop(columns = 'status', inplace=True)


## BASIC CLEANING
Before moving to modeling, the dataset needs a few consistency checks so each transaction line can be trusted as a valid customer purchase record.

First, rows with missing `Customer_ID` are removed. These rows cannot be reliably linked to a customer, so they are not useful for customer-level analysis such as churn, CLV, or retention modeling.

Next, the relationship between `Customer_ID` and `Country` is checked. In a clean transactional dataset, the same customer should usually belong to one country. If a customer appears under multiple countries, the code identifies those cases and resolves them by assigning the most representative country for that customer. The chosen country is the one linked to the highest number of invoices, and if there is a tie, the one with the highest total quantity.

After that, the code checks whether each invoice belongs to only one customer. A single invoice should normally map to a single `Customer_ID`. If one invoice is linked to multiple customer IDs, those cases are flagged for review.

Finally, the `Description` column is standardized. The same `StockCode` should generally refer to the same product name, but inconsistent descriptions can appear in raw retail data. To fix this, the most common description for each `StockCode` is kept and assigned back to the dataframe.

These cleaning steps make the dataset more consistent and help ensure that later analyses are based on stable customer, product, and transaction definitions.

In [14]:
###  Checking if customer and country mismatches

# Creating new dataframe by dropping rows with blank customer_ID
rows_with_customer = df[df['Customer_ID'].notna()].copy()

# Count unique Country per Customer_ID
customer_country_check = (
    rows_with_customer
    .groupby('Customer_ID')
    .agg(
        n_countries=('Country', lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

# Customers linked to more than one country
country_mismatch = customer_country_check[
    customer_country_check['n_countries'] > 1
]

# Save all original rows belonging to those multiple country customers
customer_country_mismatches = (
    rows_with_customer[
        rows_with_customer['Customer_ID'].isin(country_mismatch['Customer_ID'])
    ]
    .sort_values(['Customer_ID', 'Country', 'InvoiceDate'])
)

if country_mismatch.shape[0] == 0:
    print('No mismatch between Customer_ID and Country')
else:
    print(f'❌ Country column of {country_mismatch.shape[0]} customers are not unique between different lines. \n \n ID\'s of the customres are:\n')
    print(customer_country_mismatches[
    ['Customer_ID', 'Country']
].drop_duplicates().sort_values(['Customer_ID', 'Country']).head(50))

❌ Country column of 4 customers are not unique between different lines. 
 
 ID's of the customres are:

        Customer_ID      Country
12445         12417      BELGIUM
363611        12417        SPAIN
39703         12422    AUSTRALIA
204075        12422  SWITZERLAND
464323        12423      BELGIUM
320056        12423      DENMARK
110334        12431    AUSTRALIA
57086         12431      BELGIUM


In [15]:
### Resolving mismatch between country and customer

# Invoice count and total quantity by Customer_ID
customer_country_stats = (
    rows_with_customer
    .groupby(['Customer_ID', 'Country'])
    .agg(
        n_invoices=('Invoice', 'nunique'),
        total_quantity=('Quantity', 'sum')
    )
    .reset_index()
)

# Pick mode country:
# 1. highest number of invoices
# 2. if tied, highest total quantity
mode_country_per_customer = (
    customer_country_stats
    .sort_values(
        ['Customer_ID', 'n_invoices', 'total_quantity'],
        ascending=[True, False, False]
    )
    .drop_duplicates('Customer_ID', keep='first')
    [['Customer_ID', 'Country']]
    .rename(columns={'Country': 'mode_country'})
)

# Add mode country to original df
df = df.merge(
    mode_country_per_customer,
    on='Customer_ID',
    how='left'
)

# Replace Country only for rows with given Customer_ID
df.loc[df['Customer_ID'].notna(), 'Country'] = df.loc[
    df['Customer_ID'].notna(),
    'mode_country'
]

# Drop helper column
df = df.drop(columns='mode_country')

print('Mismatches resolved between Customer_ID and Country, by updating country column. ✅')

Mismatches resolved between Customer_ID and Country, by updating country column. ✅


In [16]:
### Checking whether rows corresponds to a same invoice referes to a same Customer_ID 

# Count unique Customer_ID  per Invoice
# dropna=False means missing Customer_ID is counted as its own value
invoice_check = (
    df.groupby('Invoice')
      .agg(
          n_customers=('Customer_ID', lambda x: x.nunique(dropna=False))
      )
      .reset_index()
)

# Invoices where same invoice has different customer 
bad_invoices = invoice_check[(invoice_check['n_customers'] > 1)]

# Save all original rows belonging to those problematic invoices
invoice_exceptions = (
    df[df['Invoice'].isin(bad_invoices['Invoice'])]
    .sort_values(['Invoice', 'Customer_ID'])
)


if bad_invoices.shape[0] == 0:
    print('No mismatch found between invoice and Customer_ID.  ✅')
else:
    print(f'{bad_invoices.shape[0]} mismatches observed between invoice and Customer_ID. Some mismatches as examples are:')
    print(invoice_exceptions[['Invoice', 'Customer_ID', 'Country']].drop_duplicates().head(5))

No mismatch found between invoice and Customer_ID.  ✅


In [17]:
# Number of duplicate rows (considering all columns)
n_duplicates = df.duplicated().sum()

if n_duplicates == 0:
    print("✅ No duplicate rows found.")
else:
    print(f"❌ Found {n_duplicates:,} duplicate rows.")

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)

    print(f"Deleted {n_duplicates:,} duplicate rows.")
    print(f"Remaining rows: {len(df):,}")

❌ Found 13,805 duplicate rows.
Deleted 13,805 duplicate rows.
Remaining rows: 484,985


```text 
(Invoice, StockCode) combination supposed be unique across the data. We found 5,726 duplicate (Invoice, StockCode) pairs.
 This behavior is expected in transaction-level retail data because an invoice may contain multiple line items for the same product due to separate entries, adjustments, or ERP order-line recording. Since customer value analysis is performed at the invoice level, these records will be retained and aggregated in the next steps.
```

In [18]:
# Total rows
n_rows = len(df)

# Number of unique (Invoice, StockCode) pairs
n_unique = df[['Invoice', 'StockCode']].drop_duplicates().shape[0]

print(f"Total rows: {n_rows:,}")
print(f"Unique (Invoice, StockCode) pairs: {n_unique:,}")

if n_rows == n_unique:
    print("✅ Every (Invoice, StockCode)  is unique.")
else:
    print(f"❌ Found {n_rows - n_unique:,} duplicate (Invoice, StockCode) pairs.")

Total rows: 484,985
Unique (Invoice, StockCode) pairs: 478,907
❌ Found 6,078 duplicate (Invoice, StockCode) pairs.


In [19]:
df["line_amount"] = df["Quantity"] * df["Price"]

df = (
    df.groupby(["Invoice", "StockCode"], as_index=False)
      .agg(
          Quantity=("Quantity", "sum"),
          line_amount=("line_amount", "sum"),
          Description=("Description", "first"),
          InvoiceDate=("InvoiceDate", "first"),
          **{"Customer_ID": ("Customer_ID", "first")},
          Country=("Country", "first")
      )
)

df["Price"] = df["line_amount"] / df["Quantity"]

# Total rows
n_rows = len(df)

# Number of unique (Invoice, StockCode) pairs
n_unique = df[['Invoice', 'StockCode']].drop_duplicates().shape[0]

if n_rows == n_unique:
    print("After cleaning, (Invoice, StockCode)  is unique for every lines. ✅")
    print(f"Total lines after handling duplicates is: {n_unique:,}")
else:
    print(f"❌ Found {n_rows - n_unique:,} duplicate (Invoice, StockCode) pairs from total rows: {n_rows:,}.")

After cleaning, (Invoice, StockCode)  is unique for every lines. ✅
Total lines after handling duplicates is: 478,907


In [20]:
# Number of rows before removing rows with missing Customer_ID
print(f"Rows before removing rows with missing Customer_ID : {len(df)}")

df_large = df.copy()
# Remove rows with missing Customer_ID
df = df[df["Customer_ID"].notna()].copy()

print(f"Rows remaining                                     : {len(df)}")

Rows before removing rows with missing Customer_ID : 478907
Rows remaining                                     : 381765


In [21]:
df.info()

<class 'pandas.DataFrame'>
Index: 381765 entries, 0 to 478906
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      381765 non-null  str           
 1   StockCode    381765 non-null  str           
 2   Quantity     381765 non-null  float64       
 3   line_amount  381765 non-null  float64       
 4   Description  381765 non-null  str           
 5   InvoiceDate  381765 non-null  datetime64[us]
 6   Customer_ID  381765 non-null  Int64         
 7   Country      381765 non-null  str           
 8   Price        381765 non-null  float64       
dtypes: Int64(1), datetime64[us](1), float64(3), str(4)
memory usage: 29.5 MB


In [22]:
df.head(3)

,Invoice,StockCode,Quantity,line_amount,Description,InvoiceDate,Customer_ID,Country,Price
0,489434,21232,24.0,30.0,STRAWBERRY CERAMIC TRINKET BOX,2009-12-01 07:45:00,13085,UNITED KINGDOM,1.25
1,489434,21523,10.0,59.5,FANCY FONT HOME SWEET HOME DOORMAT,2009-12-01 07:45:00,13085,UNITED KINGDOM,5.95
2,489434,21871,24.0,30.0,SAVE THE PLANET MUG,2009-12-01 07:45:00,13085,UNITED KINGDOM,1.25


## CREATING TABLES

In [23]:
print('df_large')
df_large.info()

df_large
<class 'pandas.DataFrame'>
RangeIndex: 478907 entries, 0 to 478906
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      478907 non-null  str           
 1   StockCode    478907 non-null  str           
 2   Quantity     478907 non-null  float64       
 3   line_amount  478907 non-null  float64       
 4   Description  478907 non-null  str           
 5   InvoiceDate  478907 non-null  datetime64[us]
 6   Customer_ID  381765 non-null  Int64         
 7   Country      478907 non-null  str           
 8   Price        478907 non-null  float64       
dtypes: Int64(1), datetime64[us](1), float64(3), str(4)
memory usage: 33.3 MB


In [24]:
tmp = df_large.groupby("StockCode").agg(minprice=("Price","min"), maxprice=("Price","max"), avgprice=("Price","mean"), std_of_price=("Price","std"))
desc = df_large.groupby(["StockCode","Description"]).size().reset_index(name="n").sort_values(["StockCode","n"], ascending=[True,False]).drop_duplicates("StockCode").set_index("StockCode")[["Description"]]
df_product = desc.join(tmp).reset_index()
df_product["std_of_price"] = df_product["std_of_price"].fillna(0)
df_product.head()

df_large.drop(columns='Description', inplace=True)

df_large.to_csv('calibration_large.csv', index=False)
df_product.to_csv('catalog_calib.csv', index=False)

In [25]:
df_product.head(3)

,StockCode,Description,minprice,maxprice,avgprice,std_of_price
0,10002,INFLATABLE POLITICAL GLOBE,0.72,1.70,0.962509,0.293700
1,10002R,ROBOT PENCIL SHARPNER,5.06,5.17,5.133333,0.063509
2,10080,GROOVY CACTUS INFLATABLE,0.85,0.85,0.850000,0.000000


In [26]:
df.info()

<class 'pandas.DataFrame'>
Index: 381765 entries, 0 to 478906
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      381765 non-null  str           
 1   StockCode    381765 non-null  str           
 2   Quantity     381765 non-null  float64       
 3   line_amount  381765 non-null  float64       
 4   Description  381765 non-null  str           
 5   InvoiceDate  381765 non-null  datetime64[us]
 6   Customer_ID  381765 non-null  Int64         
 7   Country      381765 non-null  str           
 8   Price        381765 non-null  float64       
dtypes: Int64(1), datetime64[us](1), float64(3), str(4)
memory usage: 29.5 MB


In [27]:
df.drop(columns=['Description'],inplace=True)
df.to_csv('calibration.csv', index=False)

In [28]:
## creating invoice table: Group by customer and invoice, then sum the costs and quantities
df_invoice = df.groupby(['Invoice', 'Customer_ID','InvoiceDate', 'Country']).agg(
    total_amount=('line_amount', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()


In [29]:
# make a copy
df_order = df_invoice.copy()

# extract date only from date-time
df_order["order_date"] = df_order["InvoiceDate"].dt.date

# group same customer + same day into one order
df_order = (
    df_order
    .groupby(["Customer_ID", "order_date"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        number_of_items=("number_of_items", "sum"),
        Country=("Country", "first"),
        InvoiceDate=('InvoiceDate', 'first')
    )
)

# create order_id as string: Customer_ID + date
df_order["order_id"] = (
    df_order["Customer_ID"].astype(str) + "_" +
    pd.to_datetime(df_order["order_date"]).dt.strftime("%Y-%m-%d")
)

df_order.drop(columns='InvoiceDate', inplace=True)

df_order['order_date'] = pd.to_datetime(df_order['order_date'])

In [30]:
print('df_order')
df_order.info()

df.to_csv('calibration.csv', index=False)
df_order.to_csv('calib_order_table.csv', index=False)

df_order
<class 'pandas.DataFrame'>
RangeIndex: 16367 entries, 0 to 16366
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype        
---  ------           --------------  -----        
 0   Customer_ID      16367 non-null  Int64        
 1   order_date       16367 non-null  datetime64[s]
 2   total_amount     16367 non-null  float64      
 3   number_of_items  16367 non-null  float64      
 4   Country          16367 non-null  str          
 5   order_id         16367 non-null  str          
dtypes: Int64(1), datetime64[s](1), float64(2), str(2)
memory usage: 783.3 KB


# CLEANING HOLDOUT DATA

In [31]:
# Holdout datasets
df = df_full[df_full["InvoiceDate"] > split_date].copy()

>**Handling cancelled order**

In [32]:
# negative price also mean a cancellation of order; 
# Since in this note, we standardize cancellation of ordre by -ve Quantity, we follow that convention
df['Quantity'] = df['Quantity'] * np.sign(df['Price'])

In [33]:
# Sort by time
df = df.sort_values("InvoiceDate").reset_index(drop=True)

# Status column
df["status"] = pd.NA

# ---------------------------------------------------
# Matching structure:
# key = (Customer_ID, StockCode, abs(Quantity), Price)
# value = list of unmatched positive row positions
# ---------------------------------------------------
open_sales = defaultdict(list)

# Use tqdm for progress tracking
for i, row in tqdm(
    enumerate(df.itertuples(index=False)),
    total=len(df),
    desc="Matching returns"
):
    key = (
        row.Customer_ID,
        row.StockCode,
        abs(row.Quantity),
        row.Price
    )
    if row.Quantity > 0:
        open_sales[key].append(i)

    elif row.Quantity < 0:
        if open_sales[key]:
            match_i = open_sales[key].pop()
            df.at[match_i, "status"] = "cancelled"
            df.at[i, "status"] = "return: purchase found"
        else:
            df.at[i, "status"] = "return: purchase not found"

# Any remaining positive rows are completed purchases
df.loc[(df["Quantity"] > 0) & (df["status"].isna()), "status"] = "completed"

df['status'] = df['status'].astype(str)

## Keeping only completed orders
df = df[df['status']=='completed']
#df_return = df[df['status'] == 'return: purchase found']

# droping status column before saving to csv
df.drop(columns = 'status', inplace=True)
#df_return.drop(columns = 'status', inplace=True)

Matching returns:   0%|          | 0/546353 [00:00<?, ?it/s]

>**Handling column mismatch**

In [34]:
### Chacking whether rows corresponds to a same invoice referes to a same Customer_ID 

# Count unique Customer_ID  per Invoice
# dropna=False means missing Customer_ID is counted as its own value
invoice_check = (
    df.groupby('Invoice')
      .agg(
          n_customers=('Customer_ID', lambda x: x.nunique(dropna=False))
      )
      .reset_index()
)

# Invoices where same invoice has different customer 
bad_invoices = invoice_check[(invoice_check['n_customers'] > 1)]

# Save all original rows belonging to those problematic invoices
invoice_exceptions = (
    df[df['Invoice'].isin(bad_invoices['Invoice'])]
    .sort_values(['Invoice', 'Customer_ID'])
)


if bad_invoices.shape[0] == 0:
    print('No mismatch found between invoice and Customer_ID.  ✅')
else:
    print(f'{bad_invoices.shape[0]} mismatches observed between invoice and Customer_ID. Some mismatches as examples are:')
    print(invoice_exceptions[['Invoice', 'Customer_ID', 'Country']].drop_duplicates().head(5))

No mismatch found between invoice and Customer_ID.  ✅


>**Handling duplicates**

In [35]:
# Number of duplicate rows (considering all columns)
n_duplicates = df.duplicated().sum()

if n_duplicates == 0:
    print("✅ No duplicate rows found.")
else:
    print(f"❌ Found {n_duplicates:,} duplicate rows.")

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)

    print(f"Deleted {n_duplicates:,} duplicate rows.")
    print(f"Remaining rows: {len(df):,}")

❌ Found 19,694 duplicate rows.
Deleted 19,694 duplicate rows.
Remaining rows: 512,646


In [36]:
df["line_amount"] = df["Quantity"] * df["Price"]

In [37]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer_ID', 'Country', 'line_amount'],
      dtype='str')

In [38]:
df_product_holdout = df.groupby(["StockCode","Description"]).size().reset_index(name="n").sort_values(["StockCode","n"], ascending=[True,False]).drop_duplicates("StockCode")[["StockCode","Description"]]

# Remove StockCodes that exist in df_product
df_product_holdout = df_product_holdout[~df_product_holdout["StockCode"].isin(df_product["StockCode"])]

# View your updated, isolated holdout set
df_product_holdout.head()

df_product_holdout.to_csv('catalog_new.csv', index=False)

In [39]:
df_product_holdout.info()

<class 'pandas.DataFrame'>
Index: 652 entries, 51 to 3895
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   StockCode    652 non-null    str  
 1   Description  652 non-null    str  
dtypes: str(2)
memory usage: 15.3 KB


In [40]:
df.drop(columns=['Description'], inplace=True)
df.info()
df.to_csv('holdout.csv', index=False)

<class 'pandas.DataFrame'>
RangeIndex: 512646 entries, 0 to 512645
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      512646 non-null  str           
 1   StockCode    512646 non-null  str           
 2   Quantity     512646 non-null  float64       
 3   InvoiceDate  512646 non-null  datetime64[us]
 4   Price        512646 non-null  float64       
 5   Customer_ID  383579 non-null  Int64         
 6   Country      512646 non-null  str           
 7   line_amount  512646 non-null  float64       
dtypes: Int64(1), datetime64[us](1), float64(3), str(3)
memory usage: 31.8 MB


>**From invoice to order**

In [41]:
## creating invoice table: Group by customer and invoice, then sum the costs and quantities
df_invoice = df.groupby(['Invoice', 'Customer_ID','InvoiceDate', 'Country']).agg(
    total_amount=('line_amount', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()

In [42]:
# make a copy
df_order = df_invoice.copy()

# extract date only from date-time
df_order["order_date"] = df_order["InvoiceDate"].dt.date

# group same customer + same day into one order
df_order = (
    df_order
    .groupby(["Customer_ID", "order_date"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        number_of_items=("number_of_items", "sum"),
        Country=("Country", "first"),
        InvoiceDate=('InvoiceDate', 'first')
    )
)

# create order_id as string: Customer_ID + date
df_order["order_id"] = (
    df_order["Customer_ID"].astype(str) + "_" +
    pd.to_datetime(df_order["order_date"]).dt.strftime("%Y-%m-%d")
)

df_order.drop(columns='InvoiceDate', inplace=True)

df_order['order_date'] = pd.to_datetime(df_order['order_date'])

In [43]:
df_order.to_csv('holdout_order_table.csv', index=False)